In [3]:
import pandas as pd
import numpy as np
import joblib

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [2]:
movies = pd.read_csv(
    "../dataset/processed/movieverse_feature_engineered.csv"
)

print(movies.shape)

movies.head()

C:\Users\HARIOM SUTHAR\AppData\Local\Temp\ipykernel_17000\2647607979.py:1: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  movies = pd.read_csv(


(570748, 18)


,movie_id,title,release_year,tagline,overview,runtime_minutes,avg_rating,genres,main_cast,poster_url,director,producer,languages,themes,countries,studios,decade,tags
0,1000001,Barbie,2023.0,She's everything. He's just Ken.,barbie and ken are having the time of their li...,114.0,3.86,comedy adventure,margot robbie as barbie ryan gosling as ken am...,https://a.ltrbxd.com/resized/film-poster/2/7/7...,greta gerwig,Tom Ackerley | Margot Robbie | Robbie Brenner,english,humanity and the world around us crude humor a...,uk usa,luckychap entertainment heyday films nb gg pic...,2020.0,comedy adventure comedy adventure comedy adven...
1,1000002,Parasite,2019.0,Act like you own the place.,all unemployed ki taek s family takes peculiar...,133.0,4.56,comedy thriller drama,song kang ho as kim ki taek lee sun kyun as pa...,https://a.ltrbxd.com/resized/film-poster/4/2/6...,bong joon ho,Jang Young-hwan | Kwak Sin-ae | Moon Yang-kwon,korean english german,humanity and the world around us intense viole...,south korea,barunson e a,2010.0,comedy thriller drama comedy thriller drama co...
2,1000003,Everything Everywhere All at Once,2022.0,The universe is so much bigger than you realize.,an aging chinese immigrant is swept up in an i...,140.0,4.30,science fiction adventure comedy action,michelle yeoh as evelyn wang ke huy quan as wa...,https://a.ltrbxd.com/resized/film-poster/4/7/4...,daniel scheinert daniel kwan,Joe Russo | Anthony Russo | Jonathan Wang,english cantonese chinese,humanity and the world around us moving relati...,usa,iac films agbo ley line entertainment,2020.0,science fiction adventure comedy action scienc...
3,1000004,Fight Club,1999.0,Mischief. Mayhem. Soap.,a ticking time bomb insomniac and a slippery s...,139.0,4.27,drama,edward norton as narrator brad pitt as tyler d...,https://a.ltrbxd.com/resized/film-poster/5/1/5...,david fincher,Ross Grayson Bell | Ceán Chaffin | Art Linson,english,intense violence and sexual transgression huma...,germany usa,fox 2000 pictures regency enterprises the lins...,1990.0,drama drama drama david fincher david fincher ...
4,1000005,La La Land,2016.0,Here's to the fools who dream.,mia an aspiring actress serves lattes to movie...,129.0,4.09,drama comedy music romance,ryan gosling as sebastian emma stone as mia jo...,https://a.ltrbxd.com/resized/film-poster/2/4/0...,damien chazelle,Jordan Horowitz | Fred Berger | Marc Platt,english,song and dance humanity and the world around u...,hong kong usa,summit entertainment black label media gilbert...,2010.0,drama comedy music romance drama comedy music ...


In [4]:
movies = movies[
    [
        "movie_id",
        "title",
        "release_year",
        "poster_url",
        "avg_rating",
        "genres",
        "languages",
        "countries",
        "director",
        "main_cast",
        "overview",
        "tags"
    ]
].copy()

In [5]:
movies.head()

,movie_id,title,release_year,poster_url,avg_rating,genres,languages,countries,director,main_cast,overview,tags
0,1000001,Barbie,2023.0,https://a.ltrbxd.com/resized/film-poster/2/7/7...,3.86,comedy adventure,english,uk usa,greta gerwig,margot robbie as barbie ryan gosling as ken am...,barbie and ken are having the time of their li...,comedy adventure comedy adventure comedy adven...
1,1000002,Parasite,2019.0,https://a.ltrbxd.com/resized/film-poster/4/2/6...,4.56,comedy thriller drama,korean english german,south korea,bong joon ho,song kang ho as kim ki taek lee sun kyun as pa...,all unemployed ki taek s family takes peculiar...,comedy thriller drama comedy thriller drama co...
2,1000003,Everything Everywhere All at Once,2022.0,https://a.ltrbxd.com/resized/film-poster/4/7/4...,4.30,science fiction adventure comedy action,english cantonese chinese,usa,daniel scheinert daniel kwan,michelle yeoh as evelyn wang ke huy quan as wa...,an aging chinese immigrant is swept up in an i...,science fiction adventure comedy action scienc...
3,1000004,Fight Club,1999.0,https://a.ltrbxd.com/resized/film-poster/5/1/5...,4.27,drama,english,germany usa,david fincher,edward norton as narrator brad pitt as tyler d...,a ticking time bomb insomniac and a slippery s...,drama drama drama david fincher david fincher ...
4,1000005,La La Land,2016.0,https://a.ltrbxd.com/resized/film-poster/2/4/0...,4.09,drama comedy music romance,english,hong kong usa,damien chazelle,ryan gosling as sebastian emma stone as mia jo...,mia an aspiring actress serves lattes to movie...,drama comedy music romance drama comedy music ...


In [6]:
movies = movies.dropna(subset=["title","tags"])

movies = movies.reset_index(drop=True)

print(movies.shape)

(570740, 12)


In [7]:
vectorizer = TfidfVectorizer(

    max_features=25000,

    stop_words="english",

    ngram_range=(1,2),

    min_df=2

)

In [8]:
tfidf_matrix = vectorizer.fit_transform(
    movies["tags"]
)

tfidf_matrix.shape

(570740, 25000)

In [9]:
knn = NearestNeighbors(

    metric="cosine",

    algorithm="brute"

)

knn.fit(tfidf_matrix)

print("Model Ready")

Model Ready


In [10]:
movie_index = pd.Series(

    movies.index,

    index=movies["title"].str.lower()

)

movie_index = movie_index[

    ~movie_index.index.duplicated()

]

In [16]:
def recommend(

    movie_name,

    start=0,

    limit=20

):

    movie_name = movie_name.lower()

    if not movie_name:
        return "Movie not found."

    if movie_name not in movie_index:
        return "Movie not found."

    idx = movie_index[movie_name]

    distances, indices = knn.kneighbors(

        tfidf_matrix[idx],

        n_neighbors=start+limit+1

    )

    indices = indices[0][1:]

    distances = distances[0][1:]

    recommendations = movies.iloc[indices].copy()

    recommendations["similarity"] = 1 - distances

    return recommendations.iloc[start:start+limit][

        [

            "title",

            "release_year",

            "avg_rating",

            "genres",

            "similarity",

            "poster_url"

        ]

    ]

In [18]:
recommend("Barbie")

,title,release_year,avg_rating,genres,similarity,poster_url
670,Swiss Army Man,2016.0,3.66,drama fantasy comedy adventure romance,0.324742,https://a.ltrbxd.com/resized/sm/upload/pp/ya/j...
1381,It's Such a Beautiful Day,2012.0,4.38,comedy animation,0.292019,https://a.ltrbxd.com/resized/sm/upload/on/mj/o...
1955,Stranger Than Fiction,2006.0,3.61,drama fantasy romance comedy,0.285200,https://a.ltrbxd.com/resized/film-poster/5/1/1...
11163,The Midnight Gospel,2020.0,4.33,mystery animation comedy,0.276632,https://a.ltrbxd.com/resized/film-poster/7/9/4...
77952,Guilt,2019.0,3.54,comedy crime drama,0.276012,https://a.ltrbxd.com/resized/film-poster/5/7/5...
4758,Wristcutters: A Love Story,2006.0,3.61,fantasy drama romance comedy,0.274667,https://a.ltrbxd.com/resized/film-poster/4/4/0...
5552,Joe Versus the Volcano,1990.0,3.18,romance comedy,0.262819,https://a.ltrbxd.com/resized/sm/upload/ve/x4/y...
3082,The Fisher King,1991.0,3.86,drama comedy,0.261634,https://a.ltrbxd.com/resized/sm/upload/px/zw/7...
179,Amélie,2001.0,4.17,romance comedy,0.258950,https://a.ltrbxd.com/resized/sm/upload/ly/0v/e...
2,Everything Everywhere All at Once,2022.0,4.30,science fiction adventure comedy action,0.257481,https://a.ltrbxd.com/resized/film-poster/4/7/4...


In [13]:
recommend("Interstellar")

,title,release_year,avg_rating,genres,similarity,poster_url
3839,Solaris,2002.0,3.25,mystery science fiction drama,0.479169,https://a.ltrbxd.com/resized/film-poster/5/0/5...
5448,Aniara,2018.0,3.54,science fiction drama,0.467531,https://a.ltrbxd.com/resized/film-poster/4/2/6...
988,Moon,2009.0,3.86,science fiction drama,0.443856,https://a.ltrbxd.com/resized/sm/upload/1z/15/k...
1603,Contact,1997.0,3.78,science fiction mystery drama,0.441785,https://a.ltrbxd.com/resized/film-poster/5/1/4...
7037,Vesper,2022.0,3.12,science fiction drama adventure fantasy,0.433533,https://a.ltrbxd.com/resized/film-poster/8/7/8...
2627,On the Silver Globe,1988.0,4.02,drama science fiction,0.419127,https://a.ltrbxd.com/resized/film-poster/3/4/8...
948,Solaris,1972.0,4.24,mystery science fiction drama,0.418524,https://a.ltrbxd.com/resized/film-poster/5/1/5...
5797,K-PAX,2001.0,3.28,science fiction mystery drama,0.417632,https://a.ltrbxd.com/resized/film-poster/5/1/8...
6803,The Quiet Earth,1985.0,3.49,drama science fiction thriller,0.416983,https://a.ltrbxd.com/resized/film-poster/4/6/4...
108,2001: A Space Odyssey,1968.0,4.30,adventure mystery science fiction,0.416338,https://a.ltrbxd.com/resized/sm/upload/fe/jp/4...


In [14]:
recommend("The Avengers")

,title,release_year,avg_rating,genres,similarity,poster_url
199,Avengers: Age of Ultron,2015.0,3.10,action adventure science fiction,0.763866,https://a.ltrbxd.com/resized/film-poster/8/1/6...
42,Avengers: Endgame,2019.0,3.87,science fiction action adventure,0.602495,https://a.ltrbxd.com/resized/film-poster/2/2/6...
39,Avengers: Infinity War,2018.0,3.96,action science fiction adventure,0.578336,https://a.ltrbxd.com/resized/film-poster/2/2/6...
152,Captain America: Civil War,2016.0,3.57,action science fiction adventure,0.470588,https://a.ltrbxd.com/resized/film-poster/1/9/4...
297,Iron Man 2,2010.0,2.87,adventure science fiction action,0.427501,https://a.ltrbxd.com/resized/film-poster/4/6/5...
99,Thor: Ragnarok,2017.0,3.73,adventure action science fiction,0.396097,https://a.ltrbxd.com/resized/film-poster/2/0/9...
1380,Transformers: Rise of the Beasts,2023.0,2.67,action science fiction adventure,0.394583,https://a.ltrbxd.com/resized/film-poster/5/9/2...
139,Iron Man,2008.0,3.70,adventure science fiction action,0.380014,https://a.ltrbxd.com/resized/film-poster/5/0/8...
785,Transformers,2007.0,3.12,action science fiction adventure,0.377879,https://a.ltrbxd.com/resized/sm/upload/5z/cu/k...
1843,Transformers: Age of Extinction,2014.0,2.01,science fiction adventure action,0.374409,https://a.ltrbxd.com/resized/film-poster/7/6/5...


In [19]:
joblib.dump(

    vectorizer,

    "../models/tfidf_vectorizer.pkl"

)

['../models/tfidf_vectorizer.pkl']

In [20]:
joblib.dump(

    knn,

    "../models/recommendation_model.pkl"

)

['../models/recommendation_model.pkl']

In [21]:
movie_index = pd.Series(
    movies.index,
    index=movies["title"].str.lower()
)

In [22]:

joblib.dump(
    movie_index,
    "../models/movie_index.pkl"
)

['../models/movie_index.pkl']

In [23]:
joblib.dump(tfidf_matrix, "../models/tfidf_matrix.pkl")

['../models/tfidf_matrix.pkl']

In [24]:
joblib.dump(
    tfidf_matrix,
    "../models/tfidf_matrix.pkl"
)

print("TF-IDF Matrix Saved")

TF-IDF Matrix Saved
